# Agentes desde cero: un coding agent

Vamos a construir, sin frameworks, un agente que encuentra y arregla bugs en un proyecto Python.

> **Agent = Model + Harness.** El modelo decide *qué* hacer. El harness (tools + loop) es el código que lo deja actuar.

```text
usuario ──► LLM ──► ¿pidió una tool? ──sí──► ejecutarla ──► resultado ──┐
             ▲            │ no                                          │
             │            ▼                                             │
             │        respuesta final                                   │
             └──────────────────────────────────────────────────────────┘
```

## 1. Instalación

In [ ]:
%pip install -U "openai>=2.45,<3" python-dotenv

## 2. API key y modelo

Usamos el `.env` del root del repo (`OPENAI_API_KEY` y, opcional, `OPENAI_MODEL`).

In [ ]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(Path("..") / ".env")

client = OpenAI()
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
print("Modelo:", OPENAI_MODEL)

## 3. El proyecto con bugs

`reset_workspace()` crea `workspace/` con una calculadora que tiene 3 bugs y sus tests. Los tests fallan.

In [ ]:
from proyecto_demo import WORKSPACE, reset_workspace, run_tests

reset_workspace()
print((WORKSPACE / "calculadora.py").read_text())
print(run_tests()[-40:])

## 4. Sin tools, el modelo solo puede hablar

No ve los archivos ni puede editarlos. Esto es un chatbot, no un agente.

In [ ]:
response = client.responses.create(
    model=OPENAI_MODEL,
    input="Los tests de mi proyecto fallan. Arreglalos.",
)
print(response.output_text)

## 5. Tools: funciones Python comunes

Le damos al agente 4 acciones sobre el workspace. Si algo falla, devolvemos el error como texto: el modelo lo lee y se corrige.

In [ ]:
def list_files() -> str:
    return "\n".join(p.name for p in WORKSPACE.iterdir() if p.suffix in (".py", ".md"))


def read_file(path: str) -> str:
    return (WORKSPACE / path).read_text()


def edit_file(path: str, old_str: str, new_str: str) -> str:
    file = WORKSPACE / path
    content = file.read_text()
    if old_str not in content:
        return "Error: old_str no está en el archivo."
    file.write_text(content.replace(old_str, new_str, 1))
    return "Archivo editado."


# run_tests ya viene de proyecto_demo: corre `python -m unittest -v` en el workspace.

print(list_files())

## 6. Describir las tools para el modelo

El modelo no ve nuestro código: ve **nombre + descripción + parámetros** (JSON Schema). Esa descripción es parte del prompt.

In [ ]:
def function_tool(name, description, params=()):
    return {
        "type": "function",
        "name": name,
        "description": description,
        "parameters": {
            "type": "object",
            "properties": {p: {"type": "string"} for p in params},
            "required": list(params),
        },
    }


TOOLS = [
    function_tool("list_files", "Lista los archivos del proyecto."),
    function_tool("read_file", "Devuelve el contenido de un archivo.", ["path"]),
    function_tool("edit_file", "Reemplaza old_str (texto exacto) por new_str en un archivo.", ["path", "old_str", "new_str"]),
    function_tool("run_tests", "Corre los tests del proyecto y devuelve la salida."),
]

TOOL_FUNCTIONS = {
    "list_files": list_files,
    "read_file": read_file,
    "edit_file": edit_file,
    "run_tests": run_tests,
}

## 7. El modelo *pide* una tool

Con `tools=TOOLS`, la respuesta puede traer un `function_call`. **El modelo no ejecuta nada**: solo dice qué quiere ejecutar.

In [ ]:
SYSTEM_PROMPT = (
    "Sos un coding agent. Leé el código antes de editarlo, "
    "corré los tests después de cada cambio y no modifiques los tests salvo que te lo pidan. "
    "Al terminar, explicá brevemente qué cambiaste."
)

response = client.responses.create(
    model=OPENAI_MODEL,
    instructions=SYSTEM_PROMPT,
    input="¿Qué archivos tiene el proyecto?",
    tools=TOOLS,
)

for item in response.output:
    print(item.type, getattr(item, "name", ""), getattr(item, "arguments", ""))

## 8. El loop del agente

Esta función **es** el agente. Todos los frameworks hacen una versión de esto:

1. Mandamos la conversación + tools al modelo.
2. Si no pidió tools → terminó.
3. Si pidió tools → las ejecutamos y le devolvemos los resultados.
4. Volvemos a 1 (con un máximo de pasos).

In [ ]:
def run_agent(task: str, max_steps: int = 15):
    conversation = [{"role": "user", "content": task}]
    tokens = []

    for step in range(1, max_steps + 1):
        response = client.responses.create(
            model=OPENAI_MODEL,
            instructions=SYSTEM_PROMPT,
            input=conversation,
            tools=TOOLS,
        )
        tokens.append(response.usage.input_tokens)
        conversation += response.output

        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:
            print(f"\n✅ Terminó en {step} pasos.\n\n{response.output_text}")
            return tokens

        for call in calls:
            args = json.loads(call.arguments)
            result = TOOL_FUNCTIONS[call.name](**args)
            print(f"[paso {step}] 🔧 {call.name}({call.arguments[:80]})")
            conversation.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": str(result),
            })

    print("⛔ Se alcanzó el máximo de pasos.")
    return tokens

## 9. Correr el agente

Le damos una tarea de alto nivel. El modelo decide qué leer, qué editar y cuándo verificar.

In [ ]:
reset_workspace()
tokens = run_agent("Los tests fallan. Encontrá los bugs y arreglalos.")

In [ ]:
# No le creemos al agente: verificamos nosotros.
print(run_tests()[-40:])

## 10. Tokens, tokens, tokens

En cada paso mandamos **toda** la conversación otra vez. El contexto crece en cada vuelta del loop: por eso los agentes son caros.

In [ ]:
for i, t in enumerate(tokens, start=1):
    print(f"paso {i:>2}: {t:>5} tokens  " + "█" * (t // 100))
print("Total:", sum(tokens))

## 11. En la terminal

`coding_agent.py` es este mismo loop con una interfaz para ver cada operación, diffs y permisos:

```bash
cd clase_5
python coding_agent.py
```

## Ejercicios

1. **Una tool nueva.** Agregá `search_code(text)` que busque un texto en los `.py` del workspace. Preguntale al agente *"¿Dónde se usa promedio?"*.
2. **Memoria.** `run_agent` arranca de cero cada vez. Hacé que reciba la `conversation` anterior y la continúe.
3. **Romper el agente.** Cambiá la descripción de `edit_file` por *"Edita archivos"*. ¿Qué pasa? ¿Cuántos pasos más necesita?

## Resumen

```text
agente = modelo + tools + loop
```

El modelo decide, las tools actúan y el loop repite hasta terminar. En el notebook 02 vemos cómo LangChain y Deep Agents nos dan este loop ya armado.